# Embedding scikit-learn toy datasets

This notebook applies `SplineGraphEmbedding` to several scikit-learn toy distributions. Labels are used only for coloring the plots; the graph fitting and embedding are unsupervised. The right-hand panels show the graph coordinates `(route_id, position)` returned by the model.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import (
    make_blobs,
    make_circles,
    make_classification,
    make_gaussian_quantiles,
    make_moons,
)

working_dir = Path.cwd().resolve()
notebooks_dir = working_dir / 'notebooks' if (working_dir / 'notebooks' / '__init__.py').exists() else working_dir
project_root = notebooks_dir.parent
if not (project_root / 'topological_graph_embedding').exists():
    raise RuntimeError('Start Jupyter from the repository root or its notebooks/ directory')
sys.path.insert(0, str(project_root))
sys.path.insert(0, str(notebooks_dir))

from topological_graph_embedding import SplineGraphEmbedding
from topological_graph_embedding.visualization.plots import plot_embedding_row

In [ ]:
toy_datasets = {}

toy_datasets['moons'] = make_moons(n_samples=500, noise=0.07, random_state=0)
toy_datasets['circles'] = make_circles(n_samples=500, factor=0.42, noise=0.045, random_state=1)
toy_datasets['blobs'] = make_blobs(
    n_samples=500,
    centers=[(-1.2, -0.8), (0.0, 1.0), (1.2, -0.4)],
    cluster_std=[0.22, 0.28, 0.20],
    random_state=2,
)
toy_datasets['classification'] = make_classification(
    n_samples=500,
    n_features=2,
    n_redundant=0,
    n_informative=2,
    n_clusters_per_class=1,
    class_sep=1.25,
    flip_y=0.04,
    random_state=3,
)
toy_datasets['gaussian-quantiles'] = make_gaussian_quantiles(
    n_samples=500,
    n_features=2,
    n_classes=3,
    random_state=4,
)

toy_datasets.keys()

In [ ]:
models = {}
embeddings = {}
summary = []

for index, (name, (X, labels)) in enumerate(toy_datasets.items()):
    model = SplineGraphEmbedding(
        n_centroids=32,
        persistence_threshold=None,
        spline_smoothing=0.1,
        max_cycles=4,
        random_state=10 + index,
    )
    result = model.fit_transform(X)
    models[name] = model
    embeddings[name] = result
    summary.append({
        'dataset': name,
        'cycles': model.realized_cycle_count_,
        'junctions': len(model.junctions_),
        'endpoints': len(model.endpoints_),
        'spline_chains': len(model.routes_),
        'median_residual': float(np.median(result.residual_norm)),
    })

summary

In [ ]:
fig, axes = plt.subplots(len(toy_datasets), 4, figsize=(26, 4 * len(toy_datasets)))
if len(toy_datasets) == 1:
    axes = axes[None, :]

for row, (name, (X, labels)) in enumerate(toy_datasets.items()):
    model = models[name]
    result = embeddings[name]
    plot_embedding_row(
        axes[row], X, labels, model, result,
        projected_title=f'{name}: data and spline graph',
        graph_title=f'{name}: graph embedding',
        metro_lines_title=f'{name}: metro-map lines',
        metro_points_title=f'{name}: metro-map points',
        jitter_seed=row,
    )

fig.suptitle('Scikit-learn toy datasets embedded with spline routes', fontsize=16, y=0.995)
fig.tight_layout()
figure_dir = project_root / 'notebooks' / 'figures'
figure_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(figure_dir / 'sklearn_toy_datasets.png', dpi=160, bbox_inches='tight')
plt.show()

In [ ]:
columns = ['dataset', 'cycles', 'junctions', 'endpoints', 'spline_chains', 'median_residual']
table_values = [[row[column] for column in columns] for row in summary]
summary_fig, summary_axis = plt.subplots(figsize=(12, 3.0))
summary_axis.axis('off')
table = summary_axis.table(cellText=table_values, colLabels=columns, loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 1.5)
summary_fig.tight_layout()
summary_fig.savefig(figure_dir / 'sklearn_toy_summary.png', dpi=160, bbox_inches='tight')
plt.show()

## Interactive toy-dataset exploration

Select one dataset, adjust the graph-fitting parameters, and refit its four-panel visualization.

In [ ]:
from html import escape
from io import BytesIO

import ipywidgets as widgets
from IPython.display import display

toy_dataset_selector = widgets.Dropdown(
    options=list(toy_datasets), value='moons', description='dataset',
)
toy_centroid_slider = widgets.IntSlider(
    value=32, min=8, max=64, step=4, description='centroids', continuous_update=False,
)
toy_smoothing_slider = widgets.FloatSlider(
    value=0.1, min=0.0, max=0.20, step=0.005, readout_format='.3f',
    description='smoothing', continuous_update=False,
)
toy_cycles_slider = widgets.IntSlider(
    value=4, min=0, max=8, step=1, description='max cycles', continuous_update=False,
)
toy_threshold_mode = widgets.Dropdown(
    options=[('automatic', 'auto'), ('manual', 'manual')],
    value='auto', description='H1 threshold',
)
toy_threshold_slider = widgets.FloatSlider(
    value=0.25, min=0.0, max=1.5, step=0.025, readout_format='.3f',
    description='manual value', continuous_update=False,
)
toy_dispersion_slider = widgets.FloatSlider(
    value=0.02, min=0.0, max=0.20, step=0.005, readout_format='.3f',
    description='dispersion', continuous_update=False,
)
toy_fit_button = widgets.Button(description='Refit selected dataset', button_style='primary')
toy_plot = widgets.Image(format='png')
toy_plot.layout.width = '100%'
toy_metrics = widgets.HTML()
last_toy_render_key = [None]

def refit_toy_dataset(_=None):
    render_key = (
        toy_dataset_selector.value,
        toy_centroid_slider.value,
        toy_smoothing_slider.value,
        toy_cycles_slider.value,
        toy_threshold_mode.value,
        toy_threshold_slider.value,
        toy_dispersion_slider.value,
    )
    if render_key == last_toy_render_key[0]:
        return
    last_toy_render_key[0] = render_key

    name = toy_dataset_selector.value
    points, labels = toy_datasets[name]
    threshold = None if toy_threshold_mode.value == 'auto' else toy_threshold_slider.value
    model = SplineGraphEmbedding(
        n_centroids=toy_centroid_slider.value,
        persistence_threshold=threshold,
        spline_smoothing=toy_smoothing_slider.value,
        max_cycles=toy_cycles_slider.value,
        random_state=10 + list(toy_datasets).index(name),
    )
    result = model.fit_transform(points)
    figure, axes = plt.subplots(1, 4, figsize=(26, 5))
    plot_embedding_row(
        axes, points, labels, model, result,
        projected_title=f'{name}: data and spline graph',
        graph_title=f'{name}: graph embedding',
        metro_lines_title=f'{name}: metro-map lines',
        metro_points_title=f'{name}: metro-map points',
        jitter_seed=0,
        metro_residual_width=toy_dispersion_slider.value,
    )
    figure.tight_layout()
    image_buffer = BytesIO()
    figure.savefig(image_buffer, format='png', dpi=160, bbox_inches='tight')
    plt.close(figure)
    toy_plot.value = image_buffer.getvalue()
    metrics = {
        'cycles': model.realized_cycle_count_,
        'junctions': len(model.junctions_),
        'endpoints': len(model.endpoints_),
        'chains': len(model.routes_),
        'median_residual': float(np.median(result.residual_norm)),
    }
    toy_metrics.value = f'<pre>{escape(str(metrics))}</pre>'

display(widgets.VBox([
    widgets.HBox([toy_dataset_selector, toy_centroid_slider, toy_cycles_slider]),
    widgets.HBox([
        toy_smoothing_slider, toy_threshold_mode, toy_threshold_slider,
        toy_dispersion_slider,
    ]),
    toy_fit_button,
    toy_plot,
    toy_metrics,
]))
refit_toy_dataset()
toy_fit_button.on_click(refit_toy_dataset)